# Experiment 12: Build a LSTM Network for Named Entity Recognition (NER)

**Aim:** Build a **Bidirectional LSTM** sequence tagger that labels every word of a sentence with an entity tag (person, organisation, location, misc, or none).

**Dataset:** CoNLL-2003 English NER (Reuters news). Tags use the BIO scheme: `O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC`.

**Model:** `[word embedding || casing-feature embedding] -> BiLSTM(128) -> BiLSTM(128) -> TimeDistributed(Dense softmax)`, with padding masked out of the loss.

**Evaluation:** entity-level precision / recall / F1 (via `seqeval`), the standard NER metric.

In [ ]:
# !pip install seqeval pandas pyarrow
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter
from seqeval.metrics import classification_report, f1_score

tf.random.set_seed(3); np.random.seed(3)
MAXLEN, EPOCHS, BATCH = 60, 8, 64
print("TensorFlow", tf.__version__)

## 1. Load CoNLL-2003

In [ ]:
BASE = "https://huggingface.co/datasets/eriktks/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003/{}/0000.parquet"
dfs = {s: pd.read_parquet(BASE.format(s)) for s in ["train", "validation", "test"]}
TAGS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]   # CoNLL-2003 label order
for s, d in dfs.items(): print(s, len(d), "sentences")
r = dfs["train"].iloc[12]
print(list(zip(r.tokens, [TAGS[t] for t in r.ner_tags])))

## 2. Pre-processing
* Words are lower-cased and mapped to integer ids (words seen < 2 times in training -> `<UNK>`).
* A **casing feature** (all-lower, Capitalised, ALLCAPS, digit, punctuation, mixed) is added, because capitalisation is a strong NER clue that lower-casing would lose.
* Sentences are padded/truncated to `MAXLEN` (index 0 = padding).

In [ ]:
cnt = Counter(w.lower() for toks in dfs["train"].tokens for w in toks)
word2idx = {"<PAD>": 0, "<UNK>": 1}
for w, c in cnt.items():
    if c >= 2: word2idx[w] = len(word2idx)
VOCAB, NT = len(word2idx), len(TAGS)
print("Vocab size:", VOCAB)

CASES = ["pad", "lower", "title", "upper", "digit", "punct", "mixed"]
def casing(w):
    if w.isdigit(): return 4
    if not any(ch.isalnum() for ch in w): return 5
    if w.islower(): return 1
    if w.istitle(): return 2
    if w.isupper(): return 3
    return 6

def encode(df):
    n = len(df)
    W = np.zeros((n, MAXLEN), "int32"); C = np.zeros((n, MAXLEN), "int32"); Y = np.zeros((n, MAXLEN), "int32")
    for i, (toks, tags) in enumerate(zip(df.tokens, df.ner_tags)):
        for j, (w, t) in enumerate(zip(toks[:MAXLEN], tags[:MAXLEN])):
            W[i, j] = word2idx.get(w.lower(), 1); C[i, j] = casing(w); Y[i, j] = t
    return W, C, Y

Wtr, Ctr, Ytr = encode(dfs["train"]); Wva, Cva, Yva = encode(dfs["validation"]); Wte, Cte, Yte = encode(dfs["test"])
# padded positions get weight 0 so they don't count in the loss
sw_tr = (Wtr != 0).astype("float32")
print(Wtr.shape, " tag distribution (train):", dict(zip(TAGS, np.bincount(Ytr[Wtr != 0], minlength=NT))))

## 3. BiLSTM tagger
A bidirectional LSTM reads each sentence left-to-right and right-to-left, so the tag of a word is decided using both its left and right context (e.g. "Washington" is a person in "Mr. Washington said" but a location in "flew to Washington").

In [ ]:
def build_model():
    w_in = layers.Input((MAXLEN,), name="words"); c_in = layers.Input((MAXLEN,), name="case")
    we = layers.Embedding(VOCAB, 100, mask_zero=True)(w_in)
    ce = layers.Embedding(len(CASES), 8)(c_in)
    x = layers.Concatenate()([we, ce]); x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2))(x)
    out = layers.TimeDistributed(layers.Dense(NT, activation="softmax"))(x)
    return keras.Model([w_in, c_in], out, name="BiLSTM_NER")

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(2e-3), loss="sparse_categorical_crossentropy", weighted_metrics=["accuracy"])
model.summary()

## 4. Train

In [ ]:
hist = model.fit([Wtr, Ctr], Ytr, sample_weight=sw_tr,
                 validation_data=([Wva, Cva], Yva, (Wva != 0).astype("float32")),
                 epochs=EPOCHS, batch_size=BATCH, verbose=2)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val"); ax[1].set_title("Token accuracy (padding excluded)"); ax[1].legend()
plt.show()

## 5. Entity-level evaluation on the test set

In [ ]:
def decode(W, Y, pred, df):
    true_seq, pred_seq = [], []
    for i in range(len(df)):
        L = min(len(df.tokens.iloc[i]), MAXLEN)
        true_seq.append([TAGS[t] for t in Y[i, :L]]); pred_seq.append([TAGS[t] for t in pred[i, :L]])
    return true_seq, pred_seq

pred_te = model.predict([Wte, Cte], batch_size=256, verbose=0).argmax(-1)
true_seq, pred_seq = decode(Wte, Yte, pred_te, dfs["test"])
print(classification_report(true_seq, pred_seq, digits=4))
print("Entity-level F1 (test):", round(f1_score(true_seq, pred_seq), 4))

## 6. Try it on new sentences

In [ ]:
def ner(sentence):
    toks = sentence.split()
    W = np.zeros((1, MAXLEN), "int32"); C = np.zeros((1, MAXLEN), "int32")
    for j, w in enumerate(toks[:MAXLEN]): W[0, j] = word2idx.get(w.lower(), 1); C[0, j] = casing(w)
    p = model.predict([W, C], verbose=0)[0].argmax(-1)[:len(toks)]
    return list(zip(toks, [TAGS[t] for t in p]))

for s in ["Barack Obama visited Germany and met Angela Merkel in Berlin on Monday .",
          "Google and Microsoft opened new offices in London last year .",
          "Sachin Tendulkar scored a century for India against Australia in Mumbai ."]:
    print(" ".join(f"{w}/{t}" if t != "O" else w for w, t in ner(s)), "\n")

## Conclusion
A 2-layer BiLSTM over word + casing embeddings can recognise persons, organisations, locations and miscellaneous entities in news text, reaching a reasonable entity-level F1 after only a few epochs (state-of-the-art systems add pretrained embeddings, character-level CNN/LSTMs and a CRF layer on top, reaching ~90+ F1). Using bidirectional context and masking padded tokens are the key design choices.